In [ ]:
                                                  ###### SECCIÓN DE TRAIN ########

In [1]:
# Celda de Setup
import sagemaker
import boto3

session = sagemaker.Session()
role = sagemaker.get_execution_role()

region = session.boto_region_name
account_id = boto3.client("sts").get_caller_identity()["Account"]

bucket = session.default_bucket()

print("Region:", region)
print("Account:", account_id)
print("Bucket:", bucket)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Region: us-east-1
Account: 988261566883
Bucket: sagemaker-us-east-1-988261566883


In [2]:
# Celda para crear repositorio ECR
import botocore

ecr = boto3.client("ecr")

repo_name = "demand-forecasting-training"

try:
    ecr.create_repository(repositoryName=repo_name)
    print("Repositorio creado")
except botocore.exceptions.ClientError:
    print("Repositorio ya existe")

image_uri = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{repo_name}:latest"

print(image_uri)

Repositorio creado
988261566883.dkr.ecr.us-east-1.amazonaws.com/demand-forecasting-training:latest


In [4]:
# Login a ECR desde notebook
!aws ecr get-login-password --region {region} | \
docker login --username AWS --password-stdin {account_id}.dkr.ecr.{region}.amazonaws.com

WARNING! Your password will be stored unencrypted in /home/sagemaker-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credential-stores

Login Succeeded


In [9]:
# Build al repo
%cd ~/demand-forecasting-model

/home/sagemaker-user/demand-forecasting-model


In [22]:
# Creamos la imagen
!docker build --network sagemaker -t demand-training:latest -f src/training/Dockerfile .

DEPRECATED: The legacy builder is deprecated and will be removed in a future release.
            BuildKit is currently disabled; enable it by removing the DOCKER_BUILDKIT=0
            environment-variable.

Sending build context to Docker daemon  242.4MB
Step 1/11 : FROM python:3.12-slim
 ---> 6f90d4a79e7a
Step 2/11 : ENV PYTHONDONTWRITEBYTECODE=1
 ---> Using cache
 ---> 3ccfa5010d79
Step 3/11 : ENV PYTHONUNBUFFERED=1
 ---> Using cache
 ---> bda4bec1b4a7
Step 4/11 : WORKDIR /opt/ml/code
 ---> Using cache
 ---> b363245b817d
Step 5/11 : COPY src/training/requirements.txt /opt/ml/code/requirements.txt
 ---> Using cache
 ---> e72c9f05fa4a
Step 6/11 : RUN pip install --no-cache-dir --upgrade pip &&     pip install --no-cache-dir -r requirements.txt
 ---> Using cache
 ---> 48eec915e7de
Step 7/11 : COPY src /opt/ml/code/src
 ---> 2deb4002fa5e
Step 8/11 : RUN mkdir -p /opt/ml/input/data/train &&     mkdir -p /opt/ml/model &&     mkdir -p /opt/ml/output/data
 ---> Running in 1af28a508581
 ---

In [23]:
# Verificamos la imagen
!docker images

REPOSITORY                                                                 TAG       IMAGE ID       CREATED          SIZE
demand-training                                                            latest    8c883f32ab6c   6 seconds ago    1.09GB
988261566883.dkr.ecr.us-east-1.amazonaws.com/demand-forecasting-training   latest    c5771477d555   10 minutes ago   1.09GB
988261566883.dkr.ecr.us-east-1.amazonaws.com/demand-forecasting-training   <none>    c369b9d2855e   2 hours ago      1.09GB


In [24]:
# Taggeamos
!docker tag demand-training:latest {image_uri}

In [25]:
# Lanzamos
!docker push {image_uri}

The push refers to repository [988261566883.dkr.ecr.us-east-1.amazonaws.com/demand-forecasting-training]

5d91817f: Preparing 
41d5ab39: Preparing 
9f45e11d: Preparing 
6907549a: Preparing 
f41274c0: Preparing 
614e27ab: Preparing 
d67c34c6: Preparing 
9dab7d18: Preparing 
latest: digest: sha256:520d5a20787052758d83c08e6c4f5dc82e61cc111040dd2a3ac9b4417720d8de size: 2201


In [14]:
# Subir el CSV de entrenamiento a S3
local_train_path = "/home/sagemaker-user/demand-forecasting-model/data/prep/sales_prep.csv"

train_data_uri = session.upload_data(
    path=local_train_path,
    bucket=bucket,
    key_prefix="demand-forecasting/train"
)

print(train_data_uri)

s3://sagemaker-us-east-1-988261566883/demand-forecasting/train/sales_prep.csv


In [15]:
# Crear el Estimator
from sagemaker.estimator import Estimator

estimator = Estimator(
    image_uri=image_uri,
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=f"s3://{bucket}/demand-forecasting/output",
    sagemaker_session=session,
)

In [26]:
# Lanzar el training job
estimator.fit({"train": train_data_uri})

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: demand-forecasting-training-2026-03-09-17-52-51-917


2026-03-09 17:52:57 Starting - Starting the training job...
2026-03-09 17:53:12 Starting - Preparing the instances for training...
2026-03-09 17:53:36 Downloading - Downloading input data...
2026-03-09 17:54:11 Training - Training image download completed. Training in progress...INFO:src.training.train:Ignoring unknown arguments from SageMaker/container: ['train']
INFO:src.training.train:Arguments received:
INFO:src.training.train:  train_data_dir=/opt/ml/input/data/train
INFO:src.training.train:  train_file=sales_prep.csv
INFO:src.training.train:  target_col=item_cnt_month
INFO:src.training.train:  time_col=date_block_num
INFO:src.training.train:  model_dir=/opt/ml/model
INFO:src.training.train:  output_data_dir=/opt/ml/output/data
INFO:src.training.train:Using CSV found in train-data-dir: /opt/ml/input/data/train/sales_prep.csv
INFO:src.training.train:Loading data from /opt/ml/input/data/train/sales_prep.csv
INFO:src.training.train:Using split date: 24.0
INFO:src.training.train:Train

In [27]:
# Crear modelo Sagemaker
from sagemaker.model import Model

model = Model(
    image_uri=image_uri,
    model_data=estimator.model_data,
    role=role,
    sagemaker_session=session
)

In [ ]:
                                                               ###### SECCIÓN DE INFERENCE ########

In [29]:
# Crear imagen para inference
inference_repo_name = "demand-forecasting-inference"
inference_image_uri = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{inference_repo_name}:latest"
print(inference_image_uri)

988261566883.dkr.ecr.us-east-1.amazonaws.com/demand-forecasting-inference:latest


In [30]:
# Crear el repositorio de inferencia
import boto3
import botocore

ecr = boto3.client("ecr")

try:
    ecr.create_repository(repositoryName=inference_repo_name)
    print("Repositorio de inferencia creado")
except botocore.exceptions.ClientError as e:
    if "RepositoryAlreadyExistsException" in str(e):
        print("El repositorio de inferencia ya existe")
    else:
        raise

Repositorio de inferencia creado


In [31]:
# Volver a la base del repo
%cd ~/demand-forecasting-model

/home/sagemaker-user/demand-forecasting-model


In [32]:
# Construir la imagen de inferencia
!docker build --network sagemaker -t demand-inference:latest -f src/inference/Dockerfile .

DEPRECATED: The legacy builder is deprecated and will be removed in a future release.
            BuildKit is currently disabled; enable it by removing the DOCKER_BUILDKIT=0
            environment-variable.

Sending build context to Docker daemon  242.4MB
Step 1/11 : FROM python:3.12-slim
 ---> 6f90d4a79e7a
Step 2/11 : ENV PYTHONDONTWRITEBYTECODE=1
 ---> Using cache
 ---> 3ccfa5010d79
Step 3/11 : ENV PYTHONUNBUFFERED=1
 ---> Using cache
 ---> bda4bec1b4a7
Step 4/11 : WORKDIR /opt/ml/code
 ---> Using cache
 ---> b363245b817d
Step 5/11 : COPY src/inference/requirements.txt /opt/ml/code/requirements.txt
 ---> 85ccde36ad98
Step 6/11 : RUN pip install --no-cache-dir --upgrade pip &&     pip install --no-cache-dir -r requirements.txt
 ---> Running in aae7f8b10abb
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 7.0 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.0.1
    Uninstalling pip-25.0.1:
      Successfully uninstalled pip-25.0.1
   ━━━━━━━━━

In [33]:
# VErificación de la imagen
!docker images

REPOSITORY                                                                 TAG       IMAGE ID       CREATED          SIZE
demand-inference                                                           latest    991be54733ab   59 seconds ago   1.1GB
988261566883.dkr.ecr.us-east-1.amazonaws.com/demand-forecasting-training   latest    8c883f32ab6c   4 hours ago      1.09GB
demand-training                                                            latest    8c883f32ab6c   4 hours ago      1.09GB
988261566883.dkr.ecr.us-east-1.amazonaws.com/demand-forecasting-training   <none>    c5771477d555   4 hours ago      1.09GB
988261566883.dkr.ecr.us-east-1.amazonaws.com/demand-forecasting-training   <none>    c369b9d2855e   6 hours ago      1.09GB


In [34]:
# Taggear la imagen
!docker tag demand-inference:latest {inference_image_uri}

In [35]:
# HAcer Push a ECR
!docker push {inference_image_uri}

The push refers to repository [988261566883.dkr.ecr.us-east-1.amazonaws.com/demand-forecasting-inference]

feb0ab42: Preparing 
2d660272: Preparing 
137179fe: Preparing 
f41274c0: Preparing 
614e27ab: Preparing 
d67c34c6: Preparing 
9dab7d18: Preparing 
latest: digest: sha256:d150e759336ec5a481417bde31ffbc41fad9d2e11127aef17184ed60576d126c size: 1994


In [36]:
# Crear el objeto model para hosting
from sagemaker.model import Model

inference_model = Model(
    image_uri=inference_image_uri,
    model_data=estimator.model_data,
    role=role,
    sagemaker_session=session
)

In [43]:
# Desplegar el endpoint
predictor = inference_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large"
)

INFO:sagemaker:Creating model with name: demand-forecasting-inference-2026-03-09-22-31-41-158
INFO:sagemaker:Creating endpoint-config with name demand-forecasting-inference-2026-03-09-22-31-41-957
INFO:sagemaker:Creating endpoint with name demand-forecasting-inference-2026-03-09-22-31-41-957


-----!

In [44]:
# Se guarda el endpoint como variable
endpoint_name = "demand-forecasting-inference-2026-03-09-22-25-21-855"

In [45]:
# Se verifica que ya está en servicio
sm = boto3.client("sagemaker", region_name=region)
desc = sm.describe_endpoint(EndpointName=endpoint_name)

print(desc["EndpointStatus"])

InService


In [46]:
# Creamos manualmente el predictor

from sagemaker.predictor import Predictor
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

predictor = Predictor(
    endpoint_name=endpoint_name,
    sagemaker_session=session,
    serializer=JSONSerializer(),
    deserializer=JSONDeserializer()
)

print(predictor)

Predictor: {'endpoint_name': 'demand-forecasting-inference-2026-03-09-22-25-21-855', 'sagemaker_session': <sagemaker.session.Session object at 0x7f3a138f8800>, 'serializer': <sagemaker.base_serializers.JSONSerializer object at 0x7f3a1116c0e0>, 'deserializer': <sagemaker.base_deserializers.JSONDeserializer object at 0x7f3a1116cfb0>}


In [47]:
# Preparamos payload

import pandas as pd
sample_df = pd.read_csv("/home/sagemaker-user/demand-forecasting-model/data/prep/sales_prep.csv")
sample_payload = sample_df.drop(columns=["item_cnt_month"]).head(1)
sample_payload

,month,date_block_num,shop_id,item_id,item_category_id,avg_price
0,1,0,0,111,57,89.0


In [48]:
# Invocamos el endpoint

payload = sample_payload.to_dict(orient="records")
response = predictor.predict(payload)
print(response)

{'predictions': [1.324721097946167]}


In [ ]:
                                               ###### BORRAR PARA NO GENERAR COSTOS #####

In [49]:
# Eliminar el endpoint
predictor.delete_endpoint()

INFO:sagemaker:Deleting endpoint configuration with name: demand-forecasting-inference-2026-03-09-22-25-21-855
INFO:sagemaker:Deleting endpoint with name: demand-forecasting-inference-2026-03-09-22-25-21-855
